- Inference ทดสอบ Model หลังจาก finetune

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# โหลดโมเดลท้องถิ่น
model_path = "/content/mysql-anomaly-bert-final"  # แก้ path ให้ตรง
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)


In [ ]:
# ทำนาย log ใหม่
def predict_anomaly(log_line):
    # ดึง command (เหมือนตอน train)
    parts = log_line.split()
    text = " ".join(parts[2:]) if len(parts) >= 4 else log_line

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=-1).item()
        prob = torch.softmax(logits, dim=-1).tolist()[0]
    return "anomaly" if pred == 1 else "normal", prob

In [ ]:
log = "2025-12-20T10:39:41.931343Z       17 Query: select load_file('/etc/passwd')"
result, confidence = predict_anomaly(log)
print(f"Prediction: {result}, Confidence: {confidence}")

- หมายเหตุ:
  
  Confidence: [ความน่าจะเป็นของ normal, ความน่าจะเป็นของ anomally]

In [ ]:
log = "2025-12-20T10:15:54.750047Z       30 Query: delete from products"
result, confidence = predict_anomaly(log)
print(f"Prediction: {result}, Confidence: {confidence}")

In [ ]:
log = "2025-12-20T10:09:04.341724Z       18 Query: show databases"
result, confidence = predict_anomaly(log)
print(result)